# NB04 · Semana 8 — Transformación con dbt (silver → gold)

**Ingeniería de Datos — LCD · UCA Rosario · 2026**  
Versión: **para el alumno**

## Objetivo

Formalizar las transformaciones como **código** con dbt: pasar de bronze a **silver** (tipado y limpieza) y a **gold** (métricas para consumo), con **tests de calidad** automatizados.

> Base de la **Entrega 2 (E2)**. Este notebook orquesta comandos dbt; el proyecto dbt vive en la carpeta `dbt/` del repositorio.

## 1. Estructura del proyecto dbt

Revisamos los modelos provistos como punto de partida. Cada grupo los adapta a su dominio.

In [5]:
import os
import sys
import subprocess
from pathlib import Path
from urllib.parse import urlsplit

# Encontrar entorno.py.
carpeta = Path.cwd().resolve()

for candidata in [carpeta, *carpeta.parents]:
    if (candidata / "entorno.py").is_file():
        sys.path.insert(0, str(candidata))
        break
else:
    raise FileNotFoundError("No se encontró entorno.py.")

from entorno import DBT_DIR, BUCKET, STORAGE_OPTIONS, info

info()

# Usar la carpeta dbt definida en entorno.py.
proyecto_dbt = Path(DBT_DIR)
proyecto_dbt.mkdir(parents=True, exist_ok=True)

# Pasar la configuración a los comandos dbt.
os.environ["AWS_ACCESS_KEY_ID"] = STORAGE_OPTIONS["AWS_ACCESS_KEY_ID"]
os.environ["AWS_SECRET_ACCESS_KEY"] = STORAGE_OPTIONS["AWS_SECRET_ACCESS_KEY"]
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["MINIO_BUCKET"] = BUCKET

url = urlsplit(STORAGE_OPTIONS["AWS_ENDPOINT_URL"])
if url.scheme not in ("http", "https") or not url.netloc:
    raise ValueError("AWS_ENDPOINT_URL debe comenzar con http:// o https://")

os.environ["DBT_S3_ENDPOINT"] = url.netloc
os.environ["DBT_S3_USE_SSL"] = str(url.scheme == "https").lower()

# Ruta de DuckDB configurable; por defecto queda dentro del proyecto dbt.
os.environ["DBT_DUCKDB_PATH"] = str(proyecto_dbt / "warehouse.duckdb")

profile = """ingdatos:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: "{{ env_var('DBT_DUCKDB_PATH') }}"
      extensions:
        - httpfs
        - delta
      secrets:
        - type: s3
          key_id: "{{ env_var('AWS_ACCESS_KEY_ID') }}"
          secret: "{{ env_var('AWS_SECRET_ACCESS_KEY') }}"
          endpoint: "{{ env_var('DBT_S3_ENDPOINT') }}"
          use_ssl: "{{ env_var('DBT_S3_USE_SSL') == 'true' }}"
          url_style: path
          region: us-east-1
"""

ruta_perfil = proyecto_dbt / "profiles.yml"

# Conservar el perfil anterior si existe y es diferente.
if ruta_perfil.exists():
    anterior = ruta_perfil.read_text(encoding="utf-8")
    if anterior != profile:
        from datetime import datetime
        respaldo = proyecto_dbt / (
            f"profiles_{datetime.now():%Y%m%d_%H%M%S_%f}.bak"
        )
        respaldo.write_text(anterior, encoding="utf-8")
        print("Respaldo:", respaldo)

ruta_perfil.write_text(profile, encoding="utf-8")

print("\nProyecto dbt:", proyecto_dbt)
print("Perfil guardado:", ruta_perfil)
print("DuckDB que usaremos:", os.environ["DBT_DUCKDB_PATH"])

Entorno detectado: servidor de la cátedra  (grupo: grupo03)
  Lake / bucket   : grupo03   →   bronze en s3://grupo03/bronze/series
  PostgreSQL      : base 'warehouse', usuario 'grupo03', host 'postgres'
  Material en     : /home/jovyan/work
Respaldo: /home/jovyan/work/dbt/profiles_20261023_003431_082437.bak

Proyecto dbt: /home/jovyan/work/dbt
Perfil guardado: /home/jovyan/work/dbt/profiles.yml
DuckDB que usaremos: /home/jovyan/work/dbt/warehouse.duckdb


## 2. Escribir todos los modelos, tests y config



In [9]:
from pathlib import Path

W = Path(DBT_DIR)

for capa in ["bronze", "silver", "gold"]:
    (W / "models" / capa).mkdir(parents=True, exist_ok=True)

(W / "tests").mkdir(parents=True, exist_ok=True)

# BRONZE: leer la tabla Delta definida en la fuente lake.vatsim.
(W / "models" / "bronze" / "brz_vatsim.sql").write_text(
    """select *
from {{ source('lake', 'vatsim') }}
""",
    encoding="utf-8",
)

# SILVER: tipado, limpieza y deduplicación.
(W / "models" / "silver" / "stg_vatsim.sql").write_text(
    """with base as (
    select
        cast(cid as bigint)                   as cid,
        cast(callsign as varchar)             as callsign,
        cast(name as varchar)                 as piloto,
        cast(server as varchar)               as server,
        cast(latitude as double)              as latitude,
        cast(longitude as double)             as longitude,
        cast(altitude as integer)             as altitude_ft,
        cast(groundspeed as integer)          as groundspeed_kt,
        cast(heading as integer)              as heading,
        nullif(flight_plan_departure, '')     as origen,
        nullif(flight_plan_arrival, '')       as destino,
        nullif(flight_plan_aircraft_short, '') as aeronave,
        cast(snapshot_ts as timestamp)        as snapshot_ts
    from {{ ref('brz_vatsim') }}
    where cid is not null
      and snapshot_ts is not null
),
dedup as (
    select *,
        row_number() over (
            partition by cid, snapshot_ts
            order by snapshot_ts
        ) as rn
    from base
)
select
    cast(cid as varchar) || '-' ||
        cast(snapshot_ts as varchar) as snapshot_sk,
    cid,
    callsign,
    piloto,
    server,
    latitude,
    longitude,
    altitude_ft,
    groundspeed_kt,
    heading,
    origen,
    destino,
    aeronave,
    snapshot_ts
from dedup
where rn = 1
""",
    encoding="utf-8",
)

# GOLD: métricas de tráfico por snapshot.
(W / "models" / "gold" / "fct_trafico_snapshot.sql").write_text(
    """with por_snapshot as (
    select
        snapshot_ts,
        count(distinct cid) as pilotos_conectados,
        count(distinct cid) filter (
            where origen is null
        ) as pilotos_sin_plan,
        count(distinct origen) as aeropuertos_origen,
        count(distinct destino) as aeropuertos_destino
    from {{ ref('stg_vatsim') }}
    group by snapshot_ts
),
con_previo as (
    select *,
        lag(pilotos_conectados) over (
            order by snapshot_ts
        ) as pilotos_previo
    from por_snapshot
)
select
    snapshot_ts,
    pilotos_conectados,
    pilotos_sin_plan,
    aeropuertos_origen,
    aeropuertos_destino,
    pilotos_previo,
    case
        when pilotos_previo is null or pilotos_previo = 0 then null
        else round(
            (pilotos_conectados - pilotos_previo)
            / cast(pilotos_previo as double) * 100,
            2
        )
    end as var_pct_pilotos
from con_previo
""",
    encoding="utf-8",
)

# Definición de la fuente, documentación y tests.
(W / "models" / "schema.yml").write_text(
    """version: 2

sources:
  - name: lake
    description: "Data lake del grupo en MinIO/S3."
    tables:
      - name: vatsim
        description: "Snapshots crudos de pilotos VATSIM en formato Delta."
        meta:
          external_location: "delta_scan('s3://{{ env_var('MINIO_BUCKET') }}/bronze/vatsim')"

models:
  - name: brz_vatsim
    description: "Bronze: vista sobre los datos crudos de VATSIM."

  - name: stg_vatsim
    description: "Silver: datos tipados y deduplicados por cid y snapshot_ts."
    columns:
      - name: snapshot_sk
        description: "Clave formada por cid y snapshot_ts."
        tests: [not_null, unique]
      - name: cid
        description: "Identificador del piloto."
        tests: [not_null]
      - name: callsign
        description: "Indicativo de vuelo."
      - name: piloto
        description: "Nombre declarado del piloto."
      - name: server
        description: "Servidor VATSIM."
      - name: latitude
        description: "Latitud en grados decimales."
      - name: longitude
        description: "Longitud en grados decimales."
      - name: altitude_ft
        description: "Altitud en pies; puede ser negativa en tierra."
      - name: groundspeed_kt
        description: "Velocidad respecto al suelo en nudos."
      - name: heading
        description: "Rumbo en grados."
      - name: origen
        description: "Aeropuerto de salida; nulo si no está informado."
      - name: destino
        description: "Aeropuerto de llegada; nulo si no está informado."
      - name: aeronave
        description: "Tipo de aeronave."
      - name: snapshot_ts
        description: "Fecha y hora del snapshot."
        tests: [not_null]

  - name: fct_trafico_snapshot
    description: "Gold: métricas de tráfico por snapshot para Metabase."
    columns:
      - name: snapshot_ts
        description: "Fecha y hora del snapshot; clave de la tabla."
        tests: [not_null, unique]
      - name: pilotos_conectados
        description: "Cantidad de pilotos distintos conectados."
        tests: [not_null]
      - name: pilotos_sin_plan
        description: "Pilotos cuyo aeropuerto de origen es nulo."
      - name: aeropuertos_origen
        description: "Cantidad de aeropuertos de salida distintos."
      - name: aeropuertos_destino
        description: "Cantidad de aeropuertos de llegada distintos."
      - name: pilotos_previo
        description: "Pilotos conectados en el snapshot anterior."
      - name: var_pct_pilotos
        description: "Variación porcentual respecto del snapshot anterior."
""",
    encoding="utf-8",
)

# Tests de rango.
(W / "tests" / "assert_latitud_valida.sql").write_text(
    """select snapshot_sk, latitude
from {{ ref('stg_vatsim') }}
where latitude is not null
  and (latitude < -90 or latitude > 90)
""",
    encoding="utf-8",
)

(W / "tests" / "assert_longitud_valida.sql").write_text(
    """select snapshot_sk, longitude
from {{ ref('stg_vatsim') }}
where longitude is not null
  and (longitude < -180 or longitude > 180)
""",
    encoding="utf-8",
)

(W / "tests" / "assert_altitud_valida.sql").write_text(
    """select snapshot_sk, altitude_ft
from {{ ref('stg_vatsim') }}
where altitude_ft is not null
  and altitude_ft < -1500
""",
    encoding="utf-8",
)

# Retirar únicamente el test anterior que rechazaba toda altitud negativa.
test_anterior = W / "tests" / "assert_altitud_no_negativa.sql"
if test_anterior.exists():
    test_anterior.unlink()

# Configuración del proyecto: coincide con el perfil del ejercicio 1.
(W / "dbt_project.yml").write_text(
    """name: 'proyecto_id'
version: '1.0.0'
config-version: 2
profile: 'ingdatos'

model-paths: ["models"]
test-paths: ["tests"]
target-path: "target"
clean-targets: ["target", "dbt_packages"]

models:
  proyecto_id:
    bronze:
      +materialized: view
    silver:
      +materialized: view
    gold:
      +materialized: table
""",
    encoding="utf-8",
)

print("Modelos, fuente lake.vatsim, tests y configuración guardados en:")
print(W)
print("Esquema:", W / "models" / "schema.yml")

Modelos, fuente lake.vatsim, tests y configuración guardados en:
/home/jovyan/work/dbt
Esquema: /home/jovyan/work/dbt/models/schema.yml


## 3. Seeds (no aplica a VATSIM)

`dbt seed` carga los CSV semilla como tablas de partida (bronze) dentro de DuckDB.

In [10]:
# VATSIM no usa seeds CSV: bronze es una tabla Delta en el lake, leída con
# delta_scan vía la source lake.vatsim. 'dbt seed' no corresponde.
print("Sin seeds: bronze es Delta en el lake (ver source lake.vatsim).")

Sin seeds: bronze es Delta en el lake (ver source lake.vatsim).


In [11]:
res = subprocess.run(
    ["dbt", "build", "--profiles-dir", "."],
    cwd=DBT_DIR,
    capture_output=True,
    text=True
)

print(res.stdout)
print(res.stderr)
res.check_returncode()

00:36:34  Running with dbt=1.12.0
00:36:35  Registered adapter: duckdb=1.10.1
00:36:35  Unable to do partial parsing because saved manifest not found. Starting full parse.
00:36:36  Found 3 models, 10 data tests, 1 source, 486 macros
00:36:36  
00:36:36  Concurrency: 1 threads (target='dev')
00:36:36  
00:36:59  1 of 13 START sql view model main.brz_vatsim ................................... [RUN]
00:36:59  1 of 13 OK created sql view model main.brz_vatsim .............................. [OK in 0.08s]
00:36:59  2 of 13 START sql view model main.stg_vatsim ................................... [RUN]
00:36:59  2 of 13 OK created sql view model main.stg_vatsim .............................. [OK in 0.03s]
00:36:59  3 of 13 START test assert_altitud_valida ....................................... [RUN]
00:36:59  3 of 13 PASS assert_altitud_valida ............................................. [PASS in 0.09s]
00:36:59  4 of 13 START test assert_latitud_valida .....................................

## 4. Inspeccionar la capa gold

Consultamos el resultado final: métricas mensuales por variable, listas para serving/BI.

In [13]:
import duckdb
con = duckdb.connect(os.path.join(DBT_DIR, 'warehouse.duckdb'))
gold = con.execute(
    "select * from fct_trafico_snapshot order by snapshot_ts desc limit 10"
).fetchdf()
con.close()
print(gold)

                 snapshot_ts  pilotos_conectados  pilotos_sin_plan  \
0 2026-10-23 00:16:39.025787                 907                39   
1 2026-10-16 22:53:54.023669                1121                48   
2 2026-10-16 22:51:54.022130                1117                49   
3 2026-10-16 22:51:39.022440                1113                49   
4 2026-09-25 22:38:08.788199                1103                40   
5 2026-09-25 22:35:53.789119                1105                37   
6 2026-09-25 22:35:23.787757                1108                38   
7 2026-09-25 22:18:38.789216                1135                48   
8 2026-09-25 22:18:23.788091                1137                51   

   aeropuertos_origen  aeropuertos_destino  pilotos_previo  var_pct_pilotos  
0                 317                  326            1121           -19.09  
1                 361                  370            1117             0.36  
2                 358                  367            1113       

# Los 4 puntos fuertes 
El secret de DuckDB: la extensión delta no lee las settings s3_* ni variables de entorno, solo el secret manager. Fue lo que destrabó el acceso a MinIO.
    
View en bronze/silver, table en gold: las views se recalculan y dependen de credenciales del lake; la gold está materializada y se consulta sin tocar S3. Por eso la celda que leía silver falló sin el secret y la que leía gold no.
    
El test de altitud que “falló”: no era un bug, era la regla candidata mal planteada (los valores eran -2 a -7 ft, aviones en tierra). Ajustaste el umbral a -1500 ft con criterio físico en vez de silenciar el test. Eso es calidad de datos real.
     
Las reglas vienen del simulacro de NB03: unicidad de (cid, snapshot_ts) y no-nulos salieron de auditar un incidente de reingesta con time travel de Delta. El linaje conceptual E1→E2 es sólido.

In [14]:
import subprocess
r = subprocess.run(
    "find /home/jovyan/work/models -name '.ipynb_checkpoints' -type d -exec rm -rf {} +",
    shell=True, capture_output=True, text=True
)
print("Checkpoints borrados.")
# verificar que quedaron solo los 4 archivos reales
print(subprocess.run(["find", "/home/jovyan/work/models", "-type", "f"],
                     capture_output=True, text=True).stdout)

Checkpoints borrados.
/home/jovyan/work/models/silver/stg_vatsim.sql
/home/jovyan/work/models/schema.yml
/home/jovyan/work/models/bronze/brz_vatsim.sql
/home/jovyan/work/models/gold/fct_trafico_snapshot.sql

